## Library importing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from sklearn.metrics.pairwise import cosine_similarity
# from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split
import pickle

# Set visualization style
sns.set_theme(style="darkgrid")


## Data Loading


First, we need to import necessary functions from the script.

In [ ]:
project_path = os.path.abspath('..')
if project_path + '/src' not in sys.path:
    sys.path.append(os.path.join(project_path, 'src'))

In [ ]:
from data_loader import load_data

Now, we can call load_data function. It will read csv file and return dataframe.

In [ ]:
ratings, tags, movies, links = load_data()

now, we can check if data is loaded successfully.

In [ ]:
tags.isnull().sum()

In [ ]:
links.isnull().sum()

In [ ]:
ratings.isnull().sum()

In [ ]:
movies.isnull().sum()

Now, we can call clean_data function. It will return cleaned dataframe. It will remove duplicates and null values.

In [ ]:
from data_preprocessing import CleanData

ratings = CleanData().clean_data(ratings, None)
tags = CleanData().clean_data_empty_string(tags, ['tag'])
movies = CleanData().clean_data(movies, None)
links = CleanData().clean_data(links, ['tmdbId'])

we can also verify if data is loaded successfully.

In [ ]:
ratings.info()
tags.info()
movies.info()
links.info()

# Exploratory Data Analysis

## Data Exploration

We can check if data is loaded successfully. We can check details about the data.

In [ ]:
data.info()

In [ ]:
data.columns

In [ ]:
data.describe()

## Cleaning Data and Filing Missing Values

We can check if there's any missing data in each column.

In [ ]:
data.isnull().sum()

Next, we can call clean_data function. It will return cleaned dataframe. It will remove duplicates and null values.

In [ ]:
cleaned_data = clean_data(data)
del data

For further analysis, we need to have oversampling(SMOTE) and undersampling. We will mainly focus on Class column as it is the target column.

In [ ]:
# under_sampled_data = under_sampling(cleaned_data)
# over_sampled_data = over_sampling(cleaned_data)

## Feature Distribution

Here, we will visualize feature distribution to understand the distribution of 'amount' feature using histogram.

In [ ]:
sns.histplot(cleaned_data['Amount'], kde=True, color='red', bins=500)

We can see that most of the transactions are less than 1000. We can use undersampling to balance the data. For that, we will apply undersampling in Feature development and Model Building.

Now, we will visualize feature distribution to understand the distribution of 'time' feature using histogram.

In [ ]:
sns.histplot(cleaned_data['Time'], kde=True, color='red', bins=500)

### Time-Series Fraud Detection

The dataset contains a Time column, which represents the seconds elapsed between the transaction and the first transaction in the dataset. As we want to verify 'which time of day' fraud/non-fraud transactions occur, we will use Density Plot to visualize the distribution.

In [ ]:
# first we can convert that to hour.
cleaned_data['Hour'] = (cleaned_data['Time'] / 3600).apply(lambda x: x % 24)

plt.figure(figsize=(12,10), dpi= 80, facecolor='w', edgecolor='k')
sns.kdeplot(cleaned_data[cleaned_data['Class'] == 1]['Hour'], label='Fraud', color='red')
sns.kdeplot(cleaned_data[cleaned_data['Class'] == 0]['Hour'], color='blue', label='Normal')

plt.xlabel('Hour of Day (0=Midnight, 23=Midnight)')
plt.ylabel('Number of Transactions')
plt.title('Fraud vs. Normal Transactions by Hour(Density Plot)')
plt.legend(['Fraud', 'Normal'])
plt.show()

# we dont need to keep the hour column as it is consuming too much memory.
del cleaned_data['Hour']

We can see that most of the transactions are in the before 10 hour. Also, we can see that most of the fraud transactions are also start from 10 hour. So, just basic analysis shows us that **most of the fraud transactions takes place when most transactions are taking place**. 

**We need to keep this mind for further analysis. As we need accuracy of training model to be higher, we need to separate fraud cases from large amount of non-fraud cases.**

### Customer Segmentation for Fraud Detection

Next, we will visualize customer segmentation for fraud detection. For that, we can use clustering techniques (like KMeans) to segment customers based on transaction patterns and identify potential fraud risk groups.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

correlation_matrix = cleaned_data.corr()

# let's consider 0.3 as the threshold. so will consider clustering only if correlation is [0.1, 1], [-1, -0.1]
positive_correlation_columns = [column for column in correlation_matrix.columns if column != 'Class' and (correlation_matrix.loc['Class'][column] > 0.1)]
negative_correlation_columns = [column for column in correlation_matrix.columns if column != 'Class' and (correlation_matrix.loc['Class'][column] < -0.1)]

print("Positive Correlation Columns: ", positive_correlation_columns)
print("Negative Correlation Columns: ", negative_correlation_columns)

selected_features = list(set(positive_correlation_columns + negative_correlation_columns + ['Amount']))

# Separate fraud and non-fraud data
fraud_data = cleaned_data[cleaned_data['Class'] == 1].copy()
non_fraud_data = cleaned_data[cleaned_data['Class'] == 0].copy()


# invoking scaled_features function from script
scaled_features_fraud = scale_features(fraud_data)
scaled_features_non_fraud = scale_features(non_fraud_data)


kmeans = KMeans(n_clusters=len(selected_features), random_state=42)

# scaling only positive and negative correlation columns to avoid data leakage.
fraud_data['Cluster'] = kmeans.fit_predict(scaled_features_fraud)
non_fraud_data['Cluster'] = kmeans.fit_predict(scaled_features_non_fraud)


# PCA to reduce dimensionality
pca = PCA(n_components=2, random_state=42)
pca_result_fraud = pca.fit_transform(scaled_features_fraud)
pca_result_non_fraud = pca.fit_transform(scaled_features_non_fraud)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=pca_result_fraud[:, 0],
    y=pca_result_fraud[:, 1],
    hue=fraud_data['Cluster'],
    palette='Set2',
    legend='auto',
    s=500
)
plt.title('Customer Segmentation for Fraud Detection')
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=pca_result_non_fraud[:, 0],
    y=pca_result_non_fraud[:, 1],
    hue=non_fraud_data['Cluster'],
    palette='Set2',
    legend='auto',
    s=500
)
plt.title('Customer Segmentation for Non Fraud Detection')
plt.show()


**Analysis:**
First, we have 2 plots. One for fraud and one for non-fraud. We will find differences between them where each cluster group can determine if it is fraud or not.
- **Fraud:** 
    - Cluster 10 is *scattered across the negative side*. 
    -   Other Clusters are *scattered across the positive side*. But **not tightly clustered**.
- **Non-Fraud:**
    - Cluster 1 and 5 are *scattered across the positive side*.
    - Other Clusters are **tightly clustered in the neutral area.**

To summarize, in fraud cases clusters are not tightly clustered. On the contrary, in non-fraud cases clusters are tightly clustered in neutral to positive side. So, in non-fraud cases, there is *similar data points between clusters except 1 and 5*. But in non-fraud cases, there is variety between clusters but *similar data points for cluster 1 and 5*. This highlights customer segmentation for fraud and non-fraud cases.

We will further analyze each of the clusters in modeling section.

## Class Distribution

Here, we will visualize class distribution and understand imbalance between fraudulent and non-fraudulent transactions. For this, we will display using all 3 (cleaned_data, under_sampled_data and over_sampled_data) dataframes.

In [ ]:
# counting number of fraud and valid transactions.
print("Fraudulent transactions: ", cleaned_data[cleaned_data['Class'] == 1].shape[0])
print("Non-fraudulent transactions: ", cleaned_data[cleaned_data['Class'] == 0].shape[0])

In [ ]:
# visualize class distribution
plt.figure(figsize=(8, 6))
sns.countplot( 
    x='Class', 
    data=cleaned_data, 
    palette='GnBu', 
    width=0.4,
    hue='Class',
)
plt.title('Class Distribution of Transactions')
plt.show()

To be noted that, we used 0 as non-fraudulent and 1 as fraudulent. We can see that there is an imbalance between fraudulent and non-fraudulent transactions. So, we tried to display plots to understand the distribution of each feature.

## Correlation Matrix

We can also plot correlation matrix to understand the correlation between features.

In [ ]:
sns.set(rc={'figure.figsize':(16,12)})
sns.heatmap(correlation_matrix, annot=False, fmt='.2f', cmap='icefire')

Here in correlation matrix plot, we will focus only on class column. 
- In heatmap, 0 or neutral -> barely have any correlation -> changing them barely have any effect on the column/row.
- In heatmap, positive -> have positive correlation -> changing them will have similar effect on the column/row(if increased, effect will also be increase)(intensity will depend on how high the value is).
- In heatmap, negative -> have negative correlation -> changing them will have opposite effect on the column/row(if increased, effect will be in decrease)(intensity will depend on how high the value is).

Therefore, in the heatmap we can see that,

In [ ]:
positive = []
negative = []
neutral = []
for column in correlation_matrix.columns:
    if column == 'Cluster' or column == 'Class':
        continue
    elif correlation_matrix.loc['Class'][column] > 0.1:
        positive.append(column)
    elif correlation_matrix.loc['Class'][column] < -0.1:
        negative.append(column)
    else:
        neutral.append(column)
        
print(f'Positive Correlation: {positive}'
      f'\nNegative Correlation: {negative}'
      f'\nNeutral Correlation: {neutral}'
      )

However, this is extremely imbalanced dataset, so later we will have to use SMOTE/under sampling to handle the imbalance for accuracy.

# Feature Engineering

## Class Imbalance Handling 

We apply SMOTE for oversampling the minority class and undersampling the majority class as needed. However, under sampling may cause loss of data for our other features, so we will use over sampling for further steps.

In [ ]:
# for that we need to divide our dataset into train(80%), test(20%). With those, we will use SMOTE(over sampling) and under sampling. We can use our script for reducing coding line.


under_sample_data = under_sampling(cleaned_data, 'Class')
over_sample_data = over_sampling(cleaned_data, 'Class')

## Feature Selection and Transformation

Here, we will need to select features and transform them into simplified form for model training. Sonst, we might build a model which is time inefficient.

In [ ]:
selected_features = scale_features(over_sample_data.drop(['Class'], axis=1))

# need to retain 98% of variance
pca = PCA(n_components=0.98, random_state=42)
pca_fit = pd.DataFrame(pca.fit_transform(selected_features))

explained_variance = np.sum(pca.explained_variance_ratio_)
print(f'Explained variance ratio with PCA: {explained_variance:.2f} or {explained_variance * 100: .2f}%')

pca_fit.columns = [f'PC{i+1}' for i in range(pca_fit.shape[1])]

After running PCA, the model found that these components actually explain 98% of the total variance. This slight increase occurs because the exact number of components is chosen by the algorithm to hit or exceed the specified threshold (0.98), and in this case, it overshot slightly.

So, 98% means we have reduced complexity in data and retained most of the significant data. This simplifies model by a lot which will increase model performance.

## Feature Importance & Explainability

In [ ]:
# XGBoost for feature importance
lightgbm_model = LGBMClassifier(
    boosting_type='gbdt',
    objective='binary',
    max_depth=5,
    metric='auc',
    learning_rate=0.05,
    max_bin=200,
    subsample=0.8,
    scale_position_weight=150,
    colsample_bytree=0.8,
    verbose=0
)

X_train, X_test, y_train, y_test = train_test_split(
    over_sample_data.drop(['Class'], axis=1),
    over_sample_data['Class'],
    test_size=0.2,
    random_state=42
)

lightgbm_model.fit(X_train, y_train)

importance = pd.Series(lightgbm_model.feature_importances_, index=X_train.columns.sort_values())

# Need to explain the model
explainer = shap.TreeExplainer(lightgbm_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar")

## Transaction Risk Scoring

Here, we will create a risk score based on transaction amount and other relevant features. This can help to further capture the risk associated with each transaction.

In [ ]:
y_pred = lightgbm_model.predict(X_test)
risk_score = pd.Series(y_pred, index=X_test.index)

risk_score_category = pd.cut(
    risk_score,
    bins=[0, 0.3, 0.6, 0.9, 1],
    labels=["Low Risk", "Medium Risk", "High Risk", "Very High Risk"]
)

risk_df = pd.DataFrame({
    "Risk Score": risk_score,
    "Risk Score Category": risk_score_category
})

We need to evaluate the performance of our model.

In [ ]:
# Evaluate model performance
check_evaluation(y_test, y_pred, risk_score)

We can even draw a confusion matrix to visualize the performance of our model.

In [ ]:
# Confusion Matrix
conf_matrix = draw_confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Fraud', 'Fraud'],
            yticklabels=['Non-Fraud', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


# Model Building

# Model Evaluation 

# Conclusion

# Reference